In [4]:
#!/usr/bin/env python3
import numpy as np, time, os
from polyhedron_gravitation_MT import PolyhedronGravitation

# --- robust path setup ---
try:
    ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    # running interactively (no __file__)
    # assume current working directory is Polyhedron/python
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA = os.path.join(ROOT, "data")
os.makedirs(DATA, exist_ok=True)

In [ ]:
# --- load geometry and evaluation points ---
V  = np.loadtxt(os.path.join(DATA, "icosahedron_vertices.csv"), delimiter=",")
F  = np.loadtxt(os.path.join(DATA, "icosahedron_faces.csv"), delimiter=",", dtype=np.int32) - 1  # <-- subtract 1 here
Pts = np.loadtxt(os.path.join(DATA, "eval_points_100k_plus_vertices.csv"), delimiter=",")


print("\n--- Python benchmark ---")
print(f"Data folder: {DATA}")
print(f"Vertices: {V.shape[0]}, Faces: {F.shape[0]}, Points: {Pts.shape[0]}")

# --- build model and compute potential ---
model = PolyhedronGravitation(V, F, G=1.0, density=1.0, eps=0.0, orient_faces=True)

t0 = time.time()
U_py = model.potential(Pts, block_size=8192)
t_py = time.time() - t0

np.savetxt(os.path.join(DATA, "U_python.csv"), U_py, delimiter=",")
with open(os.path.join(DATA, "time_python.txt"), "w") as f:
    f.write(f"python_time_sec: {t_py:.6f}\n")

print(f"Computation time: {t_py:.3f} s")
print("Saved:")
print("  data/U_python.csv")
print("  data/time_python.txt")

model.close()



--- Python benchmark ---
Data folder: /Volumes/Dunendran/Programs/Banchmark/Polyhedron/data
Vertices: 12, Faces: 20, Points: 100000
Computation time: 0.248 s
Saved:
  data/U_python.csv
  data/time_python.txt


#### Summary

In [6]:
import numpy as np, pandas as pd, os, re

In [7]:
def read_time(path, key_regex=r'[:\s]([0-9]*\.?[0-9]+)'):
    """Read runtime in seconds from a .txt timing file."""
    if not os.path.exists(path):
        return np.nan
    with open(path, 'r') as f:
        s = f.read()
    m = re.search(key_regex, s)
    return float(m.group(1)) if m else np.nan

# Load Julia BigFloat reference
U_ref_path = os.path.join(DATA, "U_ref_bigfloat.csv")
if not os.path.exists(U_ref_path):
    raise FileNotFoundError("Missing U_ref_bigfloat.csv. Run Julia benchmark first.")
U_ref = np.loadtxt(U_ref_path, delimiter=",")

rows = []

def add_row(name, ufile, tfile):
    """Append a program’s timing + accuracy info."""
    u_path = os.path.join(DATA, ufile)
    t_path = os.path.join(DATA, tfile)
    if not os.path.exists(u_path):
        rows.append({"Program": name, "Time (s)": np.nan,
                     "Mean |ΔU|": np.nan, "Max |ΔU|": np.nan,
                     "N": np.nan, "Time per pt (µs/pt)": np.nan})
        return

    U = np.loadtxt(u_path, delimiter=",")
    n = min(len(U_ref), len(U))
    diff = np.abs(U_ref[:n] - U[:n])
    t_sec = read_time(t_path)
    t_per_pt = np.nan
    if not np.isnan(t_sec) and n > 0:
        t_per_pt = (t_sec * 1e6) / n  # microseconds per point

    rows.append({
        "Program": name,
        "Time (s)": t_sec,
        "Time per pt (µs/pt)": t_per_pt,
        "Mean |ΔU|": diff.mean(),
        "Max |ΔU|": diff.max(),
        "N": int(n)
    })

# --- Add each benchmark ---
add_row("Julia Float64",            "U_julia_float64_fast.csv",    "time_julia_float64_fast.txt")
add_row("Python Float64 (MT)",      "U_python.csv",           "time_python.txt")
add_row("MATLAB Float64 (Parallel)", "U_matlab_parallel.csv", "time_matlab_parallel.txt")

# Reference timing for context
rows.append({
    "Program": "Julia BigFloat (250-digit ref)",
    "Time (s)": read_time(os.path.join(DATA, "time_julia_bigfloat.txt")),
    "Time per pt (µs/pt)": np.nan,
    "Mean |ΔU|": 0.0, "Max |ΔU|": 0.0, "N": len(U_ref)
})

# --- Assemble table ---
df = pd.DataFrame(rows,
                  columns=["Program", "Time (s)", "Time per pt (µs/pt)",
                           "Mean |ΔU|", "Max |ΔU|", "N"])
df_sorted = df.sort_values(by="Time (s)", na_position="last").reset_index(drop=True)

print("\n=== Benchmark Summary ===")
print(df_sorted.to_string(index=False, float_format=lambda x: f"{x:.6g}"))

# --- Save CSV and Markdown ---
df_sorted.to_csv(os.path.join(DATA, "benchmark_summary.csv"), index=False)
with open(os.path.join(DATA, "benchmark_summary.md"), "w") as f:
    f.write(df_sorted.to_markdown(index=False, floatfmt=".6g"))

print("\nSaved:")
print("  data/benchmark_summary.csv")
print("  data/benchmark_summary.md")


=== Benchmark Summary ===
                       Program  Time (s)  Time per pt (µs/pt)   Mean |ΔU|    Max |ΔU|      N
                 Julia Float64  0.078911              0.78911 3.01819e-15 1.31006e-14 100000
           Python Float64 (MT)  0.248003              2.48003 3.21396e-15 1.26565e-14 100000
     MATLAB Float64 (Parallel)  0.588413              5.88342 4.87798e-15 1.95399e-14 100012
Julia BigFloat (250-digit ref)    44.529                  NaN           0           0 100012

Saved:
  data/benchmark_summary.csv
  data/benchmark_summary.md


In [8]:
#!/usr/bin/env python3
import os, math, csv
from decimal import Decimal, getcontext



In [9]:
#!/usr/bin/env python3
import os, re, math
import numpy as np

# --- robust path setup ---
try:
    ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA = os.path.join(ROOT, "data")
os.makedirs(DATA, exist_ok=True)

# ---------------- utilities ----------------
SUP_MINUS = "⁻"

def format_sci(x, decimals=2):
    if not np.isfinite(x) or x == 0.0:
        return f"{'00.' + '0'*decimals}×10{SUP_MINUS}0"
    exp = int(math.floor(math.log10(abs(x))))
    mant = x / (10 ** exp)
    fmt = f"{{:0{2 + 1 + decimals}.{decimals}f}}"
    return f"{fmt.format(mant)}×10{SUP_MINUS}{abs(exp)}"

def read_time(path):
    if not os.path.exists(path):
        return np.nan
    with open(path) as f:
        s = f.read()
    m = re.search(r'([0-9]*\.?[0-9]+)', s)
    return float(m.group(1)) if m else np.nan

def load_reference(DATA):
    txt_ref = os.path.join(DATA, "U_ref_bigfloat_250digits.txt")
    csv_ref = os.path.join(DATA, "U_ref_bigfloat.csv")
    if os.path.exists(txt_ref):
        with open(txt_ref) as f:
            ref = [float(l.strip()) for l in f if l.strip()]
        return np.array(ref)
    elif os.path.exists(csv_ref):
        return np.loadtxt(csv_ref, delimiter=",")
    else:
        raise FileNotFoundError("Missing BigFloat reference file.")

U_ref = load_reference(DATA)
Nref = len(U_ref)

def compute_row(code, ufile, tfile):
    u_path = os.path.join(DATA, ufile)
    t_path = os.path.join(DATA, tfile)
    if not os.path.exists(u_path):
        return None
    U = np.loadtxt(u_path, delimiter=",", dtype=np.float64)
    n = min(len(U), Nref)
    diff = np.abs(U_ref[:n] - U[:n])
    mean = diff.mean()
    rms = np.sqrt((diff**2).mean())
    t = read_time(t_path)
    tpp = (t / n) if np.isfinite(t) and n > 0 else np.nan
    return code, tpp, mean, rms

# ---------------- compute summary ----------------
rows = [
    compute_row("Python",   "U_python.csv",                   "time_python.txt"),
    compute_row("MATLAB",   "U_matlab_parallel.csv",          "time_matlab_parallel.txt"),
    compute_row("Julia_MP", "U_julia_float64_fast.csv",       "time_julia_float64_fast.txt"),
    compute_row("Julia50",  "U_julia_bigfloat_50.csv",        "time_julia_bigfloat_50.txt"),  # new row
]
rows = [r for r in rows if r is not None]

# ---------------- print + save ----------------
header_title = "=== Gravitational Potential Accuracy & Speed Summary ==="
header_ref = "Reference: Julia BigFloat (250 digits)"
header_note = "Julia 50 comparison performed at 52-digit arithmetic precision."

print(header_title)
print(header_ref)
print(header_note)
print("\nCode  runtime (s/pt)  mean(error)   RMS(error)")

for i, (name, tpp, mean, rms) in enumerate(rows):
    print(f"{i:<5}{name:10s} {format_sci(tpp,2):>12}  {format_sci(mean,2):>12}  {format_sci(rms,2):>12}")

out_path = os.path.join(DATA, "summary_runtime_accuracy_min_simple.txt")
with open(out_path, "w") as f:
    f.write(header_title + "\n")
    f.write(header_ref + "\n")
    f.write(header_note + "\n\n")
    f.write("Code  runtime (s/pt)  mean(error)   RMS(error)\n")
    for i, (name, tpp, mean, rms) in enumerate(rows):
        f.write(f"{i:<5}{name:10s} {format_sci(tpp,2):>12}  {format_sci(mean,2):>12}  {format_sci(rms,2):>12}\n")

print(f"\nSaved: {out_path}")


=== Gravitational Potential Accuracy & Speed Summary ===
Reference: Julia BigFloat (250 digits)
Julia 50 comparison performed at 52-digit arithmetic precision.

Code  runtime (s/pt)  mean(error)   RMS(error)
0    Python       02.48×10⁻6   03.21×10⁻15   03.61×10⁻15
1    MATLAB       05.88×10⁻6   04.88×10⁻15   05.80×10⁻15
2    Julia_MP     07.89×10⁻7   03.02×10⁻15   03.42×10⁻15
3    Julia50      01.40×10⁻4    00.00×10⁻0    00.00×10⁻0

Saved: /Volumes/Dunendran/Programs/Banchmark/Polyhedron/data/summary_runtime_accuracy_min_simple.txt


In [10]:
#!/usr/bin/env python3
import os, re, math
import numpy as np
from decimal import Decimal, getcontext

# --- robust path setup ---
try:
    ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA = os.path.join(ROOT, "data")
os.makedirs(DATA, exist_ok=True)

# High-precision arithmetic for error stats
getcontext().prec = 100  # plenty for 250-digit ref vs ~1e-50 deltas

# ---------------- utilities ----------------
SUP_MINUS = "⁻"

def format_sci(x, decimals=2):
    """Format like '01.49×10⁻⁴' with fixed mantissa width."""
    if not math.isfinite(x) or x == 0.0:
        return f"{'00.' + '0'*decimals}×10{SUP_MINUS}0"
    exp = int(math.floor(math.log10(abs(x))))
    mant = x / (10 ** exp)
    fmt = f"{{:0{2 + 1 + decimals}.{decimals}f}}"
    return f"{fmt.format(mant)}×10{SUP_MINUS}{abs(exp)}"

def read_time(path):
    if not os.path.exists(path):
        return np.nan
    with open(path) as f:
        s = f.read()
    m = re.search(r'([0-9]*\.?[0-9]+)', s)
    return float(m.group(1)) if m else np.nan

# -------- reference as Decimal (full precision) --------
def load_reference_dec():
    txt_ref = os.path.join(DATA, "U_ref_bigfloat_250digits.txt")
    csv_ref = os.path.join(DATA, "U_ref_bigfloat.csv")
    if os.path.exists(txt_ref):
        ref = []
        with open(txt_ref) as f:
            for line in f:
                s = line.strip()
                if s:
                    ref.append(Decimal(s))
        return ref
    elif os.path.exists(csv_ref):
        arr = np.loadtxt(csv_ref, delimiter=",")
        return [Decimal(str(v)) for v in arr.tolist()]
    else:
        raise FileNotFoundError("Missing BigFloat reference file.")

U_ref_dec = load_reference_dec()
Nref = len(U_ref_dec)

# -------- loaders for rows --------
def load_decimals_from_txt(path):
    """Load one value per line as Decimal."""
    out = []
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s:
                out.append(Decimal(s))
    return out

def load_float_csv_as_decimal(path):
    """Load CSV float64, convert via str() to Decimal (keeps printed precision)."""
    arr = np.loadtxt(path, delimiter=",", dtype=np.float64)
    return [Decimal(str(v)) for v in arr.tolist()]

def compute_row(code, ufile, tfile, as_decimal_file=False):
    """
    If as_decimal_file=True: read ufile as Decimal text (Julia50).
    Else: read CSV floats then cast to Decimal via str().
    """
    u_path = os.path.join(DATA, ufile)
    t_path = os.path.join(DATA, tfile)
    if not os.path.exists(u_path):
        return None

    if as_decimal_file:
        U_dec = load_decimals_from_txt(u_path)
    else:
        U_dec = load_float_csv_as_decimal(u_path)

    n = min(len(U_dec), Nref)
    if n == 0:
        return None

    # high-precision mean/RMS of |Δ|
    abs_sum = Decimal(0)
    sq_sum  = Decimal(0)
    for i in range(n):
        d = (U_ref_dec[i] - U_dec[i]).copy_abs()
        abs_sum += d
        sq_sum  += d * d
    mean_abs = abs_sum / Decimal(n)
    rms_abs  = (sq_sum / Decimal(n)).sqrt()

    # timings (seconds per point)
    t = read_time(t_path)
    tpp = (t / n) if np.isfinite(t) and n > 0 else np.nan

    # Convert Decimals to float for pretty sci formatting (safe: 1e-50 >> 1e-308)
    return code, tpp, float(mean_abs), float(rms_abs)

# ---------------- compute summary ----------------
rows = [
    compute_row("Python",   "U_python.csv",                   "time_python.txt",              as_decimal_file=False),
    compute_row("MATLAB",   "U_matlab_parallel.csv",          "time_matlab_parallel.txt",     as_decimal_file=False),
    compute_row("Julia_MP", "U_julia_float64_fast.csv",       "time_julia_float64_fast.txt",  as_decimal_file=False),
    compute_row("Julia50",  "U_julia_bigfloat_50.csv",        "time_julia_bigfloat_50.txt",   as_decimal_file=True),
]
rows = [r for r in rows if r is not None]

# ---------------- print + save ----------------
header_title = "=== Gravitational Potential Accuracy & Speed Summary ==="
header_ref = "Reference: Julia BigFloat (250 digits)"
header_note = "Julia 50 comparison performed at 52-digit arithmetic precision."

print(header_title)
print(header_ref)
print(header_note)
print("\nCode  runtime (s/pt)  mean(error)   RMS(error)")

for i, (name, tpp, mean, rms) in enumerate(rows):
    print(f"{i:<5}{name:10s} {format_sci(tpp,2):>12}  {format_sci(mean,2):>12}  {format_sci(rms,2):>12}")

out_path = os.path.join(DATA, "summary_runtime_accuracy_min_simple.txt")
with open(out_path, "w") as f:
    f.write(header_title + "\n")
    f.write(header_ref + "\n")
    f.write(header_note + "\n\n")
    f.write("Code  runtime (s/pt)  mean(error)   RMS(error)\n")
    for i, (name, tpp, mean, rms) in enumerate(rows):
        f.write(f"{i:<5}{name:10s} {format_sci(tpp,2):>12}  {format_sci(mean,2):>12}  {format_sci(rms,2):>12}\n")

print(f"\nSaved: {out_path}")


=== Gravitational Potential Accuracy & Speed Summary ===
Reference: Julia BigFloat (250 digits)
Julia 50 comparison performed at 52-digit arithmetic precision.

Code  runtime (s/pt)  mean(error)   RMS(error)
0    Python       02.48×10⁻6   03.21×10⁻15   03.61×10⁻15
1    MATLAB       05.88×10⁻6   04.88×10⁻15   05.80×10⁻15
2    Julia_MP     07.89×10⁻7   03.02×10⁻15   03.42×10⁻15
3    Julia50      01.40×10⁻4   08.59×10⁻17   01.15×10⁻16

Saved: /Volumes/Dunendran/Programs/Banchmark/Polyhedron/data/summary_runtime_accuracy_min_simple.txt
